# 1. Raw data 가져오기

In [2]:
import pandas as pd

df = pd.read_excel("raw_data/20250219_시사경제용어사전.xlsx")
df.head()

/opt/anaconda3/envs/krx/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,순번,주제,용어,설명
0,1,사회,0.5인 가구,싱글족 가운데 두 곳 이상에 거처를 두거나 잦은 여행과 출장 등으로 오랫동안 집을 ...
1,2,경영,1인 창조기업,"개인이 사장이면서 직원인 기업을 의미한다. 자신이 가진 '지식, 경험, 기술' 등을..."
2,3,경제,1인당 국민소득,국민소득을 총국민 수로 나눈 값. 해당 국가의 소득 수준을 보여주는 가장 대표적인 ...
3,4,과학,20-20-20 계획,"유럽연합(EU)이 2020년까지 온실가스 20% 감축, 에너지효율 20% 개선, 신..."
4,5,금융,2차 시장(Secondary Market),"2차 시장은 처음 발행된 증권, 채권 등이 거래되는 발행시장과 구분되며, 이미 발행..."


In [5]:
finance_df = df[df["주제"] == "금융"]

print(len(finance_df))
finance_df.head()

816


,순번,주제,용어,설명
4,5,금융,2차 시장(Secondary Market),"2차 시장은 처음 발행된 증권, 채권 등이 거래되는 발행시장과 구분되며, 이미 발행..."
12,13,금융,5일선,"주가의 평균치를 이어놓은 이동평균선에서 사용되는 말로, 5일선이란 5일동안의 평균주..."
16,17,금융,ABCP(Asset Backed Commercial Paper),Asset Backed Commercial Paper의 약어. 유동화전문회사(SPC...
19,20,금융,AMA(Auto Management Account),고객이 설정한 조건에 따라 상대적으로 고금리를 주는 예금이나 증권사로 자동이체·관리...
24,25,금융,"At The Money(ATM, 앳 더 머니)",‘앳 더 머니’ 상황은 옵션의 행사가격이 기초자산의 시장 가격과 동일할 때를 가리킨...


# 2. 합성 데이터 생성

In [8]:
import os
import json
import numpy as np
from openai import OpenAI
import traceback
from dotenv import load_dotenv
import os
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
import pandas as pd


# .env 파일 로드
load_dotenv('/krx/.env')

# API_KEY 값을 가져옴
openai_api_key = os.getenv('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = openai_api_key

# Upstage API 클라이언트 설정

client_gpt = OpenAI()

In [ ]:
import json

def generate_qa_from_context(context, keyword):
    """
    Generates 5 high-quality question-answer pairs from a keyword and its explaining context.
    """
    # Define context and keyword message
    context_text = f"[KEYWORD]\n{keyword}\n[CONTEXT]\n{context}"
    
    # Define prompt to ensure clear QA pair generation around the keyword and context
    prompt = f"""
    You are a skilled question generator. Using the keyword and its explaining context provided, create exactly 5 high-quality question-answer pairs in Korean. Each question should focus on expanding the reader's understanding of the keyword by using the context. Follow these structures to ensure the output is clear, informative, and detailed.

    **Keyword and Context**:
    {context_text}

    **Question Structures**:
    - **Summarization**:
        - question: Create a question that summarizes the main idea of the context related to the keyword.
        - Example Format: "이 문장의 요약은 무엇인가요?"
        - answer: Summarize the core idea of how the context explains the keyword.

    - **Topic Analysis**:
        - question: Formulate a question identifying the main topic or use of the keyword within the context.
        - Example Format: "{keyword}의 주요 사용 또는 의미는 무엇인가요?"
        - answer: Provide an answer that clarifies the main topic or role of the keyword based on the context.

    - **Detailed Explanation**:
        - question: Ask for a more detailed explanation of a specific point mentioned in the context about the keyword.
        - Example Format: "{keyword}가 사용되는 방식에 대해 자세히 설명해 주세요."
        - answer: Elaborate on a specific detail within the context that relates to the keyword.

    - **Practical Usage**:
        - question: Pose a question about how the keyword might be used in practical situations, as described in the context.
        - Example Format: "{keyword}는 실제로 어떻게 사용될 수 있나요?"
        - answer: Describe a practical usage or example from the context.

    - **Commonsense Reasoning**:
        - question: Create a question asking about the implications or importance of the keyword in a larger context.
        - Example Format: "{keyword}의 중요성은 무엇인가요?"
        - answer: Explain why the keyword is significant, based on information from the context.

    Each question and answer should follow these templates and be provided in JSON format, in Korean.
    """
    
    # Send prompt to the model
    response = client_gpt.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{'role': 'user', 'content': prompt}],
        response_format={"type": "json_object"}
    )
    
    # Process the response
    qa_output = response.choices[0].message.content
    
    return qa_output
